## 09_03a - Editing interattivo
Solo editing: generazione, RAGAS e report restano nella 09_03.
Prima di cominciare (dopo `pre_edit`): copia il sidecar con la cella di backup. Poi: cella 1a (cambia `QID`, modifica nel widget) e 1b per controllare. Le modifiche si salvano da sole nel sidecar.

In [1]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys, json, logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                    datefmt="%H:%M:%S", force=True)

QDIR = project_root / "data/queries/dst"
sidecar_path = project_root / "data/processed/dst/dst_benchmark.json"
queries_path = QDIR / "eval_queries.json"          # file unico (42 query): serve a update_query(new_id=...)
pred_path, norm_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json"

handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)

bench = RagasBenchmark(
    handler, predetermined_path=pred_path, normal_path=norm_path,
    out_dir=project_root / "data/benchmark/dst",
    top_j=8, top_k=20, top_n=7,
)
print("!>> Handler e benchmark pronti")

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


!>> Pronto


In [ ]:
### BACKUP sidecar (una volta, prima di modificare)
shutil.copy(sidecar_path, sidecar_path.with_name("dst_benchmark_pre_edit.json"))

In [2]:
### 1a ### EDITING INTERATTIVO - scegli la query: retrieval + reranking + widget (modifiche salvate nel sidecar)
import ipywidgets as widgets
from IPython.display import display

QID = "CPX_P01"# cambia id per passare alla query successiva
q = bench.prepare(QID, order_by=["DIST","TAGS"], manual_tags=True) # mostra anche i tag assegnati nei json 
print(f"[{QID}] {q['query']}\n\nREFERENCE:\n{q['reference']}")

PHASE_MAP = {"1 Retrieval (Vettoriale)": "vectorial", "2 Retrieval (Distanze)": "distance",
             "Top-J Reranked": "reranked", "All Reranked": "all_reranked"}
phase_selector = widgets.ToggleButtons(options=list(PHASE_MAP), description="Fase Grafo:", button_style="info")
output_area = widgets.Output()

def on_phase_change(change):
    with output_area:
        output_area.clear_output(wait=True)
        handler.update_widget_view(PHASE_MAP[change["new"]])
        display(handler.widget)

phase_selector.observe(on_phase_change, names="value")
handler.update_widget_view("vectorial")
display(phase_selector, output_area)
with output_area:
    display(handler.widget)

!>> Tag assegnati alla query:
   >>> Tag di Conversazione: []
   >>> Tag Manuali della Query: ['noise, distracting documents and context position']
   >>> Assegnazione Tag Automatici disabilitata
<<< Lista completa: ['noise, distracting documents and context position']
[SPE_P03] In 'The Power of Noise', how are gold, relevant, distracting and random documents defined?

REFERENCE:
The gold document is the original NQ passage containing the answer for the query. Relevant documents are passages that, like the gold one, contain the correct answer and are contextually useful; the gold document is itself a relevant document. Distracting documents are semantically or contextually linked to the topic of the query but do not contain the answer (e.g. a passage about Josephine's horse when asked about the colour of Napoleon's horse). Random documents have no relation to the query and do not contain the answer, acting as informational noise.


ToggleButtons(button_style='info', description='Fase Grafo:', options=('1 Retrieval (Vettoriale)', '2 Retrieva…

Output()

In [ ]:
### 1b ### Dopo le modifiche: rilancia reranking e confronta i chunk (e la risposta, se vuoi) con la baseline
handler.run_top_j_distance_tag_reranking()
cmp = handler.compare_retrieval_vs_reranking()
print("baseline :", cmp["vectorial_baseline"]["chunk_ids"][:bench.top_j])
print("reranked :", cmp["post_reranking"]["chunk_ids"])
display(bench.edit_status().query("n_dist_edits > 0 or n_graph_tags > 0"))